# 01-01 레이어는 공간을 바꾸는 함수예요

런타임 유형을 GPU로 바꾼 뒤 첫 셀부터 차례로 실행하세요.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

## 1. 점 25개 준비하기

In [ ]:
xs = torch.arange(-2, 3, dtype=torch.float32)  # (5,)
grid = torch.cartesian_prod(xs, xs)  # (25, 2), 각 행이 점 하나 (x0, x1)
print(grid.shape)
print(grid[:5])

## 2. 선형 레이어를 직접 만들기: y = Wx + b

In [ ]:
W = torch.tensor([[1.0, 0.8], [-0.4, 1.2]])  # (2, 2), 가중치
b = torch.tensor([1.0, 0.5])  # (2,), 편향

def linear(x):
    # x: (N, 2) -> (N, 2)
    return x @ W.T + b

out = linear(grid)  # (25, 2)
print(grid[0], "->", out[0])

In [ ]:
def draw_lines(ax, f, color, n=9, lim=2):
    # 격자선도 같은 함수에 통과시켜 공간이 어떻게 휘는지 그려요
    t = torch.linspace(-lim, lim, 50)
    for v in torch.linspace(-lim, lim, n):
        for line in (torch.stack([t, torch.full_like(t, v)], 1), torch.stack([torch.full_like(t, v), t], 1)):
            p = f(line)
            ax.plot(p[:, 0], p[:, 1], color=color, lw=0.6, alpha=0.5)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, f, pts, color, title, names in [
    (axes[0], lambda p: p, grid, "tab:blue", "input x", ("x0", "x1")),
    (axes[1], linear, out, "gold", "output y = Wx + b", ("y0", "y1")),
]:
    draw_lines(ax, f, "gray")
    ax.scatter(pts[:, 0], pts[:, 1], color=color, edgecolor="k", zorder=3)
    ax.scatter(*pts[12], color="red", s=80, zorder=4)  # 원점 (0, 0)에서 출발한 점
    ax.set_xlim(-5, 5)
    ax.set_ylim(-4, 4)
    ax.set_aspect("equal")
    ax.axhline(0, color="k", lw=0.8)
    ax.axvline(0, color="k", lw=0.8)
    ax.set_xlabel(names[0])
    ax.set_ylabel(names[1])
    ax.set_title(title)
plt.show()

## 3. 가중치를 바꾸면 변환도 바뀌어요

In [ ]:
import math

c, s = math.cos(math.pi / 6), math.sin(math.pi / 6)
examples = {
    "scale [[2,0],[0,0.5]]": torch.tensor([[2.0, 0.0], [0.0, 0.5]]),
    "rotate 30 deg": torch.tensor([[c, -s], [s, c]]),
    "shear [[1,1],[0,1]]": torch.tensor([[1.0, 1.0], [0.0, 1.0]]),
    "flatten [[1,1],[1,1]]": torch.tensor([[1.0, 1.0], [1.0, 1.0]]),
}
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, Wk) in zip(axes, examples.items()):
    f = lambda p, Wk=Wk: p @ Wk.T
    draw_lines(ax, f, "gray")
    pts = f(grid)
    ax.scatter(pts[:, 0], pts[:, 1], color="gold", edgecolor="k", zorder=3)
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.set_aspect("equal")
    ax.set_xlabel("y0")
    ax.set_ylabel("y1")
    ax.set_title(name)
plt.show()

## 4. PyTorch의 `nn.Linear`와 비교하기

In [ ]:
layer = nn.Linear(2, 2)  # 입력 2개 -> 출력 2개
print(layer.weight.shape, layer.bias.shape)  # (2, 2), (2,)

with torch.no_grad():
    layer.weight.copy_(W)
    layer.bias.copy_(b)

out_layer = layer(grid)  # (25, 2)
print(torch.allclose(out_layer, out))

## 5. 입력과 출력의 크기가 달라도 돼요

In [ ]:
layer = nn.Linear(8, 4)  # 입력 8개 -> 출력 4개
x = torch.rand(5, 8)  # (batch=5, 8)
y = layer(x)  # (5, 4)
print(x.shape, "->", y.shape)
print(layer.weight.shape)  # (4, 8) = (출력 수, 입력 수)

In [ ]:
w = layer.weight.detach()  # (4, 8)
plt.figure(figsize=(7, 3.5))
plt.imshow(w, cmap="RdBu_r", vmin=-0.4, vmax=0.4)
plt.colorbar()
plt.xlabel("input index (8)")
plt.ylabel("output index (4)")
plt.title("nn.Linear(8, 4).weight")
plt.show()

## 6. 학습된 가중치의 히트맵

손글씨 숫자(8×8 픽셀, 입력 64개)를 0~9로 분류하는 `nn.Linear(64, 10)`을 학습시킨 뒤 가중치를 그려요.
학습 코드는 [01-04](https://github.com/alsrjs0725/EntryToPytorchForStudent/blob/main/docs/01-레이어-이론/04-학습-루프로-분류-모델-만들기.md)에서 설명해요. 지금은 실행만 하고 그림을 봐요.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()  # scikit-learn에 들어 있는 8x8 손글씨 숫자 1797장
X = torch.tensor(digits.data, dtype=torch.float32) / 16  # (1797, 64), 0~1로 맞춤
t = torch.tensor(digits.target)  # (1797,), 정답 숫자

# 학습: 01-04에서 설명해요
model = nn.Linear(64, 10)  # 입력 64개 -> 숫자 10개의 점수
opt = torch.optim.Adam(model.parameters(), lr=0.05, weight_decay=1e-2)
for step in range(300):
    loss = nn.functional.cross_entropy(model(X), t)
    opt.zero_grad()
    loss.backward()
    opt.step()
print("accuracy:", (model(X).argmax(1) == t).float().mean().item())

w = model.weight.detach()  # (10, 64)
lim = w.abs().max()
fig, axes = plt.subplots(2, 10, figsize=(15, 3.6))
for d in range(10):
    axes[0, d].imshow(X[t == d].mean(0).reshape(8, 8), cmap="gray_r")  # 숫자 d의 평균 이미지
    axes[0, d].set_title(f"mean of {d}")
    im = axes[1, d].imshow(w[d].reshape(8, 8), cmap="RdBu_r", vmin=-lim, vmax=lim)  # 가중치 d번 행을 8x8로
    axes[1, d].set_title(f"weight[{d}]")
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(im, ax=axes[1].tolist(), shrink=0.9)
plt.show()

In [ ]:
# 출력 0번은 가중치 0번 행과 입력의 가중합에 편향을 더한 값이에요
first = x[0]  # (8,), 배치의 첫 번째 데이터
manual = (layer.weight[0] * first).sum() + layer.bias[0]
print(manual.item(), y[0, 0].item())